<a href="https://colab.research.google.com/github/falyseck/multilingual_qa/blob/main/multilingual_health_qa_starter_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multilingual Health Q&A — Starter Notebook

**Challenge:** Multilingual Health Question Answering in Low-Resource African Languages

This notebook provides two baselines for the challenge:

| Baseline | Approach | Speed | Quality |
|---|---|---|---|
| **Baseline 1** | TF-IDF retrieval | Very fast | Low |
| **Baseline 2** | Multilingual LLM (mT5 / NLLB) | Moderate | Higher |

**Task:** Given a health question in one of five African languages (Amharic, Luganda, Akan, Swahili, or English), generate a fluent and accurate answer in the **same language**.

**Evaluation metrics:**
- `TargetRLF1` — ROUGE-L F1 score
- `TargetR1F1` — ROUGE-1 F1 score
- `TargetLLM`  — LLM-as-a-Judge score

> **Note:** All three submission columns should contain the same generated answer for each row. The platform computes all three metrics from that single answer.

## 1 — Install and Import Packages

In [ ]:
# Install required packages
!pip install -q scikit-learn pandas numpy rouge-score
!pip install -q transformers sentencepiece accelerate torch

print('✅ Packages installed')

In [ ]:
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', None)

print('✅ Imports complete')

## 2 — Set File Paths

In [ ]:
DATA_DIR = Path('.')

TRAIN_PATH      = DATA_DIR / 'Train.csv'
TEST_PATH       = DATA_DIR / 'Test.csv'
VAL_PATH        = DATA_DIR / 'Val.csv'
SAMPLE_SUB_PATH = DATA_DIR / 'SampleSubmission.csv'

# Output paths for each baseline
OUTPUT_TFIDF = DATA_DIR / 'submission_tfidf_baseline.csv'
OUTPUT_LLM   = DATA_DIR / 'submission_llm_baseline.csv'

for path in [TRAIN_PATH, TEST_PATH, VAL_PATH, SAMPLE_SUB_PATH]:
    status = '✅' if path.exists() else '❌'
    print(f'{status} {path}')

## 3 — Load and Preview the Data

In [ ]:
train             = pd.read_csv(TRAIN_PATH)
test              = pd.read_csv(TEST_PATH)
val               = pd.read_csv(VAL_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

print(f'Train shape             : {train.shape}')
print(f'Test shape              : {test.shape}')
print(f'Val shape               : {val.shape}')
print(f'Sample submission shape : {sample_submission.shape}')
print()
print('Train columns:', train.columns.tolist())
print('Test columns :', test.columns.tolist())
print('Val columns  :', val.columns.tolist())

display(train.head(3))
display(test.head(3))
display(sample_submission.head(3))

In [ ]:
# Explore language distribution in training data
print('Language distribution in training set:')
display(train['subset'].value_counts())

In [ ]:
ID_COL           = 'ID'
TEST_ID_COL      = 'ID'
QUESTION_COL     = 'input'
TEST_QUESTION_COL= 'input'
ANSWER_COL       = 'output'
LANG_COL         = 'subset'
TEST_LANG_COL    = 'subset'

print(f'  Train ID        : {ID_COL}')
print(f'  Test ID         : {TEST_ID_COL}')
print(f'  Train question  : {QUESTION_COL}')
print(f'  Test question   : {TEST_QUESTION_COL}')
print(f'  Train answer    : {ANSWER_COL}')
# ── Language code mapping ──────────────────────────────────────────────────────
# The `subset` column encodes language and country as "<LangCode>_<CountryCode>".
# Only the first part identifies the language; the second is the country.
# This dictionary maps the language prefix to a full language name used in prompts.
SUBSET_TO_LANGUAGE = {
    'Eng': 'English',
    'Aka': 'Akan',
    'Lug': 'Luganda',
    'Swa': 'Swahili',
    'Amh': 'Amharic',
}

def subset_to_language_name(subset_code: str) -> str:
    """
    Extract the full language name from a subset code such as 'Amh_Eth' or 'Aka_Gha'.
    Falls back to the raw code if the prefix is not recognised.
    """
    if not subset_code or not isinstance(subset_code, str):
        return 'English'
    lang_prefix = subset_code.split('_')[0]
    return SUBSET_TO_LANGUAGE.get(lang_prefix, subset_code)

print('Language mapping:')
for code, name in SUBSET_TO_LANGUAGE.items():
    print(f'  {code}_* → {name}')


## 5 — Text Cleaning

In [ ]:
def clean_text(x):
    """Strip whitespace and handle null values."""
    if pd.isna(x):
        return ''
    return str(x).strip()

train[QUESTION_COL]      = train[QUESTION_COL].map(clean_text)
train[ANSWER_COL]        = train[ANSWER_COL].map(clean_text)
test[TEST_QUESTION_COL]  = test[TEST_QUESTION_COL].map(clean_text)
val[QUESTION_COL]        = val[QUESTION_COL].map(clean_text)
val[ANSWER_COL]          = val[ANSWER_COL].map(clean_text)

# Remove rows with empty questions or answers
train = train[(train[QUESTION_COL] != '') & (train[ANSWER_COL] != '')].reset_index(drop=True)
test  = test[test[TEST_QUESTION_COL]  != ''].reset_index(drop=True)
val   = val[(val[QUESTION_COL] != '') & (val[ANSWER_COL] != '')].reset_index(drop=True)

print(f'Cleaned train shape : {train.shape}')
print(f'Cleaned test shape  : {test.shape}')
print(f'Cleaned val shape   : {val.shape}')

## 7 — Evaluation Utilities

ROUGE-1 and ROUGE-L scoring using whitespace tokenisation — safe for non-English scripts.

In [ ]:
try:
    from rouge_score import rouge_scorer

    class WhitespaceTokenizer:
        """Whitespace tokeniser — language-agnostic and safe for African scripts."""
        def tokenize(self, text):
            if text is None:
                return []
            return str(text).strip().split()

    def compute_rouge(predictions, references):
        """
        Compute mean ROUGE-1 and ROUGE-L F1 scores.

        Parameters
        ----------
        predictions : list[str]
        references  : list[str]

        Returns
        -------
        dict with rouge1_f1 and rougeL_f1
        """
        scorer = rouge_scorer.RougeScorer(
            ['rouge1', 'rougeL'],
            tokenizer    = WhitespaceTokenizer(),
            use_stemmer  = False,
        )
        r1_scores, rl_scores = [], []

        for pred, ref in zip(predictions, references):
            score = scorer.score(str(ref), str(pred))
            r1_scores.append(score['rouge1'].fmeasure)
            rl_scores.append(score['rougeL'].fmeasure)

        return {
            'rouge1_f1': float(np.mean(r1_scores)) if r1_scores else 0.0,
            'rougeL_f1': float(np.mean(rl_scores)) if rl_scores else 0.0,
        }

    def compute_rouge_by_language(predictions, references, languages):
        """Compute ROUGE scores broken down by language."""
        results = {}
        lang_arr = np.array(languages)

        for lang in np.unique(lang_arr):
            mask    = lang_arr == lang
            preds_l = [p for p, m in zip(predictions, mask) if m]
            refs_l  = [r for r, m in zip(references,  mask) if m]
            results[lang] = compute_rouge(preds_l, refs_l)

        return pd.DataFrame(results).T

    print('✅ ROUGE scorer loaded')

except ImportError:
    print('⚠️  rouge-score not installed. Run: pip install rouge-score')
    compute_rouge = None

## 8 — Baseline 1: TF-IDF Retrieval

For each test question, find the most similar training question using TF-IDF character n-grams and return its answer.

**Why character n-grams?** Character-level features work across scripts (Latin, Amharic Ge'ez, etc.) without requiring language-specific tokenisation.

In [ ]:
class TfidfRetrievalAnswerer:
    """
    TF-IDF nearest-neighbour retrieval baseline.

    Builds a per-language model if a group column is available,
    falling back to a global model for unseen groups.
    """

    def __init__(self, question_col, answer_col, group_col=None,
                 ngram_range=(3, 5), max_features=200_000):
        self.question_col = question_col
        self.answer_col   = answer_col
        self.group_col    = group_col
        self.ngram_range  = ngram_range
        self.max_features = max_features
        self.models       = {}
        self.global_model = None

    def _fit_single(self, df):
        """Fit a vectoriser and nearest-neighbour index on a subset."""
        vectorizer = TfidfVectorizer(
            analyzer     = 'char_wb',
            ngram_range  = self.ngram_range,
            min_df       = 1,
            max_features = self.max_features,
            lowercase    = False,   # preserve case for non-Latin scripts
        )
        questions = df[self.question_col].fillna('').astype(str).tolist()
        answers   = df[self.answer_col].fillna('').astype(str).tolist()

        X  = vectorizer.fit_transform(questions)
        nn = NearestNeighbors(n_neighbors=1, metric='cosine')
        nn.fit(X)

        return {
            'vectorizer': vectorizer,
            'nn'        : nn,
            'answers'   : np.array(answers,   dtype=object),
            'questions' : np.array(questions, dtype=object),
        }

    def fit(self, df):
        """Fit the global model and per-group models."""
        self.global_model = self._fit_single(df)
        if self.group_col and self.group_col in df.columns:
            for group, sub in df.groupby(self.group_col):
                if len(sub) >= 2:
                    self.models[group] = self._fit_single(sub)
        print(f'  Fitted global model + {len(self.models)} group model(s)')
        return self

    def _predict_one_from_model(self, question, model):
        Xq        = model['vectorizer'].transform([question])
        dist, idx = model['nn'].kneighbors(Xq, n_neighbors=1)
        i         = idx[0][0]
        sim       = 1 - float(dist[0][0])
        return model['answers'][i], sim, model['questions'][i]

    def predict_one(self, question, group=None):
        model = self.models.get(group, self.global_model) if group is not None else self.global_model
        return self._predict_one_from_model(question, model)

    def predict(self, df, question_col, group_col=None):
        outputs, similarities, matched = [], [], []
        for _, row in df.iterrows():
            question = clean_text(row[question_col])
            group    = row[group_col] if group_col and group_col in df.columns else None
            answer, sim, matched_q = self.predict_one(question, group)
            outputs.append(answer)
            similarities.append(sim)
            matched.append(matched_q)
        return outputs, similarities, matched

print('✅ TfidfRetrievalAnswerer defined')

In [ ]:
# Choose grouping strategy — prefer config (language+country), else language
GROUP_COL      =  LANG_COL
TEST_GROUP_COL =  TEST_LANG_COL

print(f'Group column      : {GROUP_COL}')
print(f'Test group column : {TEST_GROUP_COL}')

In [ ]:
# ── Validate TF-IDF baseline on the local validation set ──────────────────────
print('Training TF-IDF retrieval baseline on training partition...')

answerer_valid = TfidfRetrievalAnswerer(
    question_col = QUESTION_COL,
    answer_col   = ANSWER_COL,
    group_col    = GROUP_COL,
).fit(train)

valid_pred, valid_sim, valid_match = answerer_valid.predict(
    val,
    question_col = QUESTION_COL,
    group_col    = GROUP_COL,
)

if compute_rouge:
    metrics = compute_rouge(valid_pred, val[ANSWER_COL].tolist())
    print(f'\n📊 TF-IDF Baseline — Validation ROUGE Scores')
    print(f'   ROUGE-1 F1 : {metrics["rouge1_f1"]:.4f}')
    print(f'   ROUGE-L F1 : {metrics["rougeL_f1"]:.4f}')

    # Break down by language
    if LANG_COL and LANG_COL in val.columns:
        print('\n📊 ROUGE scores by language:')
        lang_metrics = compute_rouge_by_language(
            valid_pred,
            val[ANSWER_COL].tolist(),
            val[LANG_COL].tolist()
        )
        display(lang_metrics.round(4))

# Preview
preview = val[[ID_COL, QUESTION_COL, ANSWER_COL]].copy()
preview['baseline_answer']        = valid_pred
preview['retrieval_similarity']   = [f'{s:.3f}' for s in valid_sim]
preview['matched_train_question'] = valid_match
display(preview.head(5))

In [ ]:
# ── Train on all data and predict test answers ─────────────────────────────────
print('Training TF-IDF retrieval on full training set...')

answerer = TfidfRetrievalAnswerer(
    question_col = QUESTION_COL,
    answer_col   = ANSWER_COL,
    group_col    = GROUP_COL,
).fit(train)

test_pred_tfidf, test_sim, test_match = answerer.predict(
    test,
    question_col = TEST_QUESTION_COL,
    group_col    = TEST_GROUP_COL,
)

print(f'Generated {len(test_pred_tfidf)} predictions')

preview = test[[TEST_ID_COL, TEST_QUESTION_COL]].copy()
preview['baseline_answer']      = test_pred_tfidf
preview['retrieval_similarity'] = [f'{s:.3f}' for s in test_sim]
display(preview.head(5))

## 9 — Baseline 2: Multilingual LLM (mT5 / AfroLM)

This baseline uses a pre-trained multilingual sequence-to-sequence model to generate answers directly, without retrieval.

**Model options (choose one):**

| Model | Languages | Size | Notes |
|---|---|---|---|
| `google/mt5-small` | 101 languages incl. Swahili | 300M | Fast, good multilingual coverage |
| `google/mt5-base` | 101 languages | 580M | Better quality, slower |
| `facebook/nllb-200-distilled-600M` | 200 languages incl. Luganda, Akan | 600M | Best African language coverage |
| `Helsinki-NLP/opus-mt-mul-en` | Many → English | 300M | English output only |

> **Recommendation:** Start with `google/mt5-small` for speed, then try `facebook/nllb-200-distilled-600M` for better coverage of low-resource languages.

**For a strong fine-tuned solution:**
Fine-tune `google/mt5-base` or `facebook/nllb-200-distilled-600M` on the provided training data using the Hugging Face `Seq2SeqTrainer`. See the fine-tuning section at the end of this notebook.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── Configuration ──────────────────────────────────────────────────────────────
# Choose a model — uncomment the one you want to use
MODEL_NAME = 'google/mt5-small'             # Fastest — good for a first run
#MODEL_NAME = 'google/mt5-base'            # Better quality
#MODEL_NAME = 'facebook/nllb-200-distilled-600M'  # Best African language coverage

MAX_INPUT_LENGTH  = 256    # Maximum tokens for input question
MAX_OUTPUT_LENGTH = 128   # Maximum tokens for generated answer
BATCH_SIZE_LLM    = 4      # Reduce to 4 if you get out-of-memory errors
NUM_BEAMS         = 8   # Beam search width — higher = better quality, slower

# Use GPU if available
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
print(f'Model   : {MODEL_NAME}')

if DEVICE == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Load the model and tokeniser ───────────────────────────────────────────────
print(f'Loading {MODEL_NAME}...')
print('This may take a few minutes on first run (downloading model weights).')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_llm = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    # Always load in float32 so gradient computation stays in float32.
    # fp16/bf16 mixed precision is handled by the Trainer via grad scaler,
    # not by storing the model weights in float16 directly.
    torch_dtype = torch.float32,
)
model_llm = model_llm.to(DEVICE)
model_llm.eval()

print(f'✅ {MODEL_NAME} loaded on {DEVICE}')
print(f'   Parameters : {sum(p.numel() for p in model_llm.parameters()) / 1e6:.0f}M')

In [ ]:
import re

def build_prompt(question: str, language: str = None) -> str:
    """
    Build an input prompt for the model.

    For mT5: prefix the question with a task description.
    The model learns to associate the prefix with the generation task.

    `language` may be a raw subset code (e.g. 'Amh_Eth') or a full language
    name. It is resolved through `subset_to_language_name` so the model always
    receives a human-readable language name in the prompt rather than an opaque
    code.

    Parameters
    ----------
    question : str
        The health question to answer.
    language : str, optional
        Subset code (e.g. 'Amh_Eth') or full language name. Resolved to a
        human-readable name before being inserted into the prompt.

    Returns
    -------
    str
    """
    if language:
        lang_name = subset_to_language_name(language)
        return f'Answer this health question in {lang_name}: {question}'
    return f'Answer this health question: {question}'



def generate_answers_batch(questions: list, languages: list = None,
                           batch_size: int = BATCH_SIZE_LLM) -> list:
    """
    Generate answers for a list of questions using the loaded LLM.

    Processes questions in batches to avoid out-of-memory errors.

    Parameters
    ----------
    questions : list[str]
    languages : list[str], optional
    batch_size : int

    Returns
    -------
    list[str]
    """
    if languages is None:
        languages = [None] * len(questions)

    all_answers = []
    n_batches   = (len(questions) + batch_size - 1) // batch_size

    for batch_idx in range(n_batches):
        start = batch_idx * batch_size
        end   = min(start + batch_size, len(questions))

        batch_questions = questions[start:end]
        batch_languages = languages[start:end]

        # Build prompts
        prompts = [
            build_prompt(q, l)
            for q, l in zip(batch_questions, batch_languages)
        ]

        # Tokenise
        inputs = tokenizer(
            prompts,
            return_tensors = 'pt',
            padding        = True,
            truncation     = True,
            max_length     = MAX_INPUT_LENGTH,
        ).to(DEVICE)

        # Generate
        with torch.no_grad():
            outputs = model_llm.generate(
                **inputs,
                max_new_tokens  = MAX_OUTPUT_LENGTH,
                num_beams       = NUM_BEAMS,
                early_stopping  = True,
                no_repeat_ngram_size = 3,
                lenghth_penalty = 1.2
            )

        # Decode
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # Post-process: strip mT5 sentinel tokens (<extra_id_N>) that the
        # model may emit when it has not been fine-tuned on a seq2seq task.
        # mT5 is pre-trained with a span-corruption objective that uses these
        # tokens as placeholders; a zero-shot prompt may trigger them because
        # the model has never been trained to suppress them in open generation.
        cleaned = [re.sub(r'<extra_id_\d+>', '', ans).strip() for ans in decoded]
        all_answers.extend(cleaned)

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == n_batches:
            print(f'  Batch {batch_idx + 1}/{n_batches} — {end}/{len(questions)} questions processed')

    return all_answers

print('✅ LLM generation functions defined')

In [ ]:
# ── Quick sanity check on a few examples ──────────────────────────────────────
print('Running sanity check on 3 validation examples...')

import re
import torch

def build_prompt(question: str, language: str = None) -> str:
    """
    Build an input prompt for the model.

    For mT5: prefix the question with a task description.
    The model learns to associate the prefix with the generation task.

    `language` may be a raw subset code (e.g. 'Amh_Eth') or a full language
    name. It is resolved through `subset_to_language_name` so the model always
    receives a human-readable language name in the prompt rather than an opaque
    code.

    Parameters
    ----------
    question : str
        The health question to answer.
    language : str, optional
        Subset code (e.g. 'Amh_Eth') or full language name. Resolved to a
        human-readable name before being inserted into the prompt.

    Returns
    -------
    str
    """
    if language:
        lang_name = subset_to_language_name(language)
        return f'Answer this health question in {lang_name}: {question}'
    return f'Answer this health question: {question}'


def generate_answers_batch(questions: list, languages: list = None,
                           batch_size: int = BATCH_SIZE_LLM) -> list:
    """
    Generate answers for a list of questions using the loaded LLM.

    Processes questions in batches to avoid out-of-memory errors.

    Parameters
    ----------
    questions : list[str]
    languages : list[str], optional
    batch_size : int

    Returns
    -------
    list[str]
    """
    if languages is None:
        languages = [None] * len(questions)

    all_answers = []
    n_batches   = (len(questions) + batch_size - 1) // batch_size

    for batch_idx in range(n_batches):
        start = batch_idx * batch_size
        end   = min(start + batch_size, len(questions))

        batch_questions = questions[start:end]
        batch_languages = languages[start:end]

        # Build prompts
        prompts = [
            build_prompt(q, l)
            for q, l in zip(batch_questions, batch_languages)
        ]

        # Tokenise
        inputs = tokenizer(
            prompts,
            return_tensors = 'pt',
            padding        = True,
            truncation     = True,
            max_length     = MAX_INPUT_LENGTH,
        ).to(DEVICE)

        # Generate
        with torch.no_grad():
            outputs = model_llm.generate(
                **inputs,
                max_new_tokens  = MAX_OUTPUT_LENGTH,
                num_beams       = NUM_BEAMS,
                early_stopping  = True,
                no_repeat_ngram_size = 3,
                length_penalty = 1.2, # FIX IS HERE

            )

        # Decode
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # Post-process: strip mT5 sentinel tokens (<extra_id_N>) that the
        # model may emit when it has not been fine-tuned on a seq2seq task.
        cleaned = [re.sub(r'<extra_id_\d+>', '', ans).strip() for ans in decoded]
        all_answers.extend(cleaned)

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == n_batches:
            print(f'  Batch {batch_idx + 1}/{n_batches} — {end}/{len(questions)} questions processed')

    return all_answers

sample     = val.head(3)
q_sample   = sample[QUESTION_COL].tolist()
l_sample   = sample[LANG_COL].tolist() if LANG_COL else None
ref_sample = sample[ANSWER_COL].tolist()

gen_sample = generate_answers_batch(q_sample, l_sample, batch_size=3)

for i, (q, ref, gen) in enumerate(zip(q_sample, ref_sample, gen_sample)):
    lang = l_sample[i] if l_sample else 'unknown'
    print(f'\n[{i+1}] Language : {lang}')
    print(f'    Question  : {q[:120]}')
    print(f'    Reference : {ref[:120]}')
    print(f'    Generated : {gen[:120]}')

In [ ]:
# ── Validate LLM baseline on the local validation set ─────────────────────────
# Note: this cell can take several minutes depending on your hardware.
# Reduce VALIDATION_SAMPLE_SIZE to speed it up during experimentation.

VALIDATION_SAMPLE_SIZE = 100   # Set to None to evaluate on the full validation set

if VALIDATION_SAMPLE_SIZE:
    val_sample = val.sample(
        n            = min(VALIDATION_SAMPLE_SIZE, len(val)),
        random_state = SEED
    )
else:
    val_sample = val

print(f'Evaluating LLM baseline on {len(val_sample)} validation examples...')

val_questions = val_sample[QUESTION_COL].tolist()
val_languages = val_sample[LANG_COL].tolist() if LANG_COL else None
val_references= val_sample[ANSWER_COL].tolist()

val_predictions_llm = generate_answers_batch(val_questions, val_languages)

if compute_rouge:
    metrics_llm = compute_rouge(val_predictions_llm, val_references)
    print(f'\n📊 LLM Baseline — Validation ROUGE Scores ({MODEL_NAME})')
    print(f'   ROUGE-1 F1 : {metrics_llm["rouge1_f1"]:.4f}')
    print(f'   ROUGE-L F1 : {metrics_llm["rougeL_f1"]:.4f}')

    if LANG_COL and LANG_COL in val_sample.columns:
        print('\n📊 ROUGE scores by language (LLM baseline):')
        lang_metrics_llm = compute_rouge_by_language(
            val_predictions_llm,
            val_references,
            val_sample[LANG_COL].tolist()
        )
        display(lang_metrics_llm.round(4))

In [ ]:
# ── Generate LLM predictions for the full test set ────────────────────────────
print(f'Generating LLM answers for {len(test)} test questions...')
print('This may take several minutes.')

test_questions_all = test[TEST_QUESTION_COL].tolist()
test_languages_all = test[TEST_LANG_COL].tolist() if TEST_LANG_COL else None

test_pred_llm = generate_answers_batch(test_questions_all, test_languages_all)

print(f'\n✅ Generated {len(test_pred_llm)} answers')

# Preview
preview_llm = test[[TEST_ID_COL, TEST_QUESTION_COL]].copy()
preview_llm['llm_answer'] = test_pred_llm
display(preview_llm.head(5))

## 10 — Create Submission Files

Each submission must contain exactly four columns: `ID`, `TargetRLF1`, `TargetR1F1`, `TargetLLM`.
All three target columns should contain the same generated answer.

In [ ]:
def make_submission(ids, predictions, output_path):
    """
    Build and save a valid Zindi submission file.

    Parameters
    ----------
    ids         : array-like of row IDs
    predictions : list[str] of generated answers
    output_path : str or Path
    """
    # Belt-and-suspenders: strip any residual sentinel tokens before saving.
    clean_preds = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in predictions]

    sub = pd.DataFrame()
    sub['ID']         = ids
    sub['TargetRLF1'] = clean_preds
    sub['TargetR1F1'] = clean_preds
    sub['TargetLLM']  = clean_preds

    sub = sub[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]

    # ── Submission checks ─────────────────────────────────────────────────
    required_cols = ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
    assert list(sub.columns) == required_cols, \
        f'Expected columns {required_cols}, got {list(sub.columns)}'
    assert len(sub) == len(test), \
        f'Row count mismatch: {len(sub)} predictions vs {len(test)} test rows'
    assert sub[['TargetRLF1', 'TargetR1F1', 'TargetLLM']].notna().all().all(), \
        'Missing values found in submission'
    assert (sub['TargetRLF1'] == sub['TargetR1F1']).all(), \
        'TargetRLF1 and TargetR1F1 differ'
    assert (sub['TargetRLF1'] == sub['TargetLLM']).all(), \
        'TargetRLF1 and TargetLLM differ'

    sub.to_csv(output_path, index=False, encoding='utf-8')
    print(f'✅ Submission saved to: {output_path}')
    print(f'   Shape : {sub.shape}')
    display(sub.head(3))
    return sub


# ── Save TF-IDF submission ─────────────────────────────────────────────────────
print('Saving TF-IDF baseline submission...')
sub_tfidf = make_submission(test[TEST_ID_COL].values, test_pred_tfidf, OUTPUT_TFIDF)

print()

# ── Save LLM submission ────────────────────────────────────────────────────────
print('Saving LLM baseline submission...')
sub_llm = make_submission(test[TEST_ID_COL].values, test_pred_llm, OUTPUT_LLM)

## 11 — Compare Baselines

In [ ]:
if compute_rouge:
    # Recompute TF-IDF validation scores for comparison
    tfidf_preds_val, _, _ = answerer_valid.predict(
        val_sample, question_col=QUESTION_COL, group_col=GROUP_COL
    )
    metrics_tfidf = compute_rouge(tfidf_preds_val, val_references)

    comparison = pd.DataFrame({
        'Baseline'  : ['TF-IDF Retrieval', f'LLM ({MODEL_NAME})'],
        'ROUGE-1 F1': [metrics_tfidf['rouge1_f1'], metrics_llm['rouge1_f1']],
        'ROUGE-L F1': [metrics_tfidf['rougeL_f1'], metrics_llm['rougeL_f1']],
    })

    print('📊 Baseline Comparison (validation set):')
    display(comparison.round(4))
    print()
    print('Note: The LLM score shown here is for a zero-shot (untrained) model.')
    print('Fine-tuning on the training data will significantly improve this.')

## 12 — Fine-tuning the LLM on Training Data

Fine-tuning adapts the pre-trained mT5 weights to the health QA task and all five languages. After training, answers are regenerated for the test set and a new submission file is saved.

**Key fixes vs. the original template:**
- Prompts are built with `build_prompt()` at training time — matching exactly what is used at inference
- Language names (resolved from `subset`) are included in training prompts
- Padding tokens in labels are masked to `-100` so the loss ignores them
- `DataCollatorForSeq2Seq` handles dynamic padding correctly
- A validation split monitors overfitting during training
- After `trainer.train()`, the test set is regenerated and a new submission is saved

In [ ]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset

# ── Fine-tuning configuration ──────────────────────────────────────────────
FINETUNE_OUTPUT_DIR     = './mt5-finetuned-health-qa'
FINETUNE_EPOCHS         = 3
FINETUNE_BATCH_SIZE     = 4      # Reduce to 4 if you hit OOM errors
FINETUNE_LEARNING_RATE  = 5e-4
FINETUNE_MAX_INPUT_LEN  = 256    # Must match MAX_INPUT_LENGTH used at inference
FINETUNE_MAX_TARGET_LEN = 128    # Must match MAX_OUTPUT_LENGTH used at inference
FINETUNE_VAL_SIZE       = 0.02  # 5% of training data used for validation

OUTPUT_FINETUNED = DATA_DIR / 'submission_finetuned_llm.csv'

print(f'Fine-tuning config:')
print(f'  Model            : {MODEL_NAME}')
print(f'  Epochs           : {FINETUNE_EPOCHS}')
print(f'  Batch size       : {FINETUNE_BATCH_SIZE}')
print(f'  Learning rate    : {FINETUNE_LEARNING_RATE}')
print(f'  Max input tokens : {FINETUNE_MAX_INPUT_LEN}')
print(f'  Max target tokens: {FINETUNE_MAX_TARGET_LEN}')
print(f'  Val split        : {FINETUNE_VAL_SIZE:.0%}')
print(f'  Output dir       : {FINETUNE_OUTPUT_DIR}')


In [ ]:
# ── Experiment 3: Subsample training data for fine-tuning only ────────────
# Experiments 1 & 2 already completed on full data — this only affects fine-tuning
SAMPLE_FRACTION = 0.05

train_finetune = (
    train
    .groupby(LANG_COL, group_keys=False)
    .apply(lambda g: g.sample(frac=SAMPLE_FRACTION, random_state=SEED))
    .reset_index(drop=True)
)

print(f"Full train set      : {len(train):,} examples  ← Exp 1 & 2 used this")
print(f"Fine-tuning sample  : {len(train_finetune):,} examples  ← Exp 3 uses this")
print(train_finetune[LANG_COL].value_counts())

In [ ]:
# ── Build prompt-aware training dataset ───────────────────────────────────
# Critical: we use build_prompt() here so training inputs match inference
# inputs exactly. The language name (resolved from subset) is included so
# the model learns to condition its output on the target language.

def make_hf_dataset(df, question_col, answer_col, lang_col):
    """
    Convert a pandas DataFrame to a HuggingFace Dataset with prompt-formatted
    inputs and tokenised labels.

    Parameters
    ----------
    df           : pd.DataFrame
    question_col : str  — column containing the question text
    answer_col   : str  — column containing the reference answer
    lang_col     : str  — column containing the subset code (e.g. 'Amh_Eth')

    Returns
    -------
    datasets.Dataset with columns: input_ids, attention_mask, labels
    """
    records = []
    for _, row in df.iterrows():
        prompt = build_prompt(
            question = str(row[question_col]),
            language = str(row[lang_col]) if lang_col and lang_col in df.columns else None,
        )
        records.append({'prompt': prompt, 'answer': str(row[answer_col])})

    raw_ds = Dataset.from_list(records)

    def preprocess(examples):
        # Tokenise inputs (prompts)
        model_inputs = tokenizer(
            examples['prompt'],
            max_length  = FINETUNE_MAX_INPUT_LEN,
            truncation  = True,
            padding     = False,   # DataCollatorForSeq2Seq handles padding dynamically
        )
        # Tokenise targets (reference answers)
        # Use text_target= (the modern API). The older context manager
        # was removed in transformers >= 4.28 and must not be used.
        labels = tokenizer(
            text_target = examples['answer'],
            max_length  = FINETUNE_MAX_TARGET_LEN,
            truncation  = True,
            padding     = False,
        )
        # Mask padding tokens in labels so the loss ignores them.
        # Without this the model wastes capacity learning to predict pad tokens.
        label_ids = labels['input_ids']
        model_inputs['labels'] = [
            [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
            for seq in label_ids
        ]
        return model_inputs

    return raw_ds.map(preprocess, batched=True, remove_columns=['prompt', 'answer'])


# ── Split off a validation set from training data ──────────────────────────
from sklearn.model_selection import train_test_split

train_df, val_ft_df = train_test_split(
    train_finetune,   # from train to train_finetune
    test_size    = FINETUNE_VAL_SIZE,
    random_state = SEED,
    stratify     = train_finetune[LANG_COL] if LANG_COL in train.columns else None,  # from train to train_finetune
)

print(f'Fine-tuning split — train: {len(train_df):,}  val: {len(val_ft_df):,}')

hf_train_ds = make_hf_dataset(train_df,  QUESTION_COL, ANSWER_COL, LANG_COL)
hf_val_ds   = make_hf_dataset(val_ft_df, QUESTION_COL, ANSWER_COL, LANG_COL)

print(f'HF train dataset : {hf_train_ds}')
print(f'HF val dataset   : {hf_val_ds}')


In [ ]:
import os
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# ── Data collator ─────────────────────────────────────────────────────────
data_collator = DataCollatorForSeq2Seq(
    tokenizer          = tokenizer,
    model              = model_llm,
    label_pad_token_id = -100,
    pad_to_multiple_of = 8,
)

# ── Training arguments ────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir                  = FINETUNE_OUTPUT_DIR,
    num_train_epochs            = FINETUNE_EPOCHS,

    # ── Memory fixes ──────────────────────────────────────────────────────
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    per_device_eval_batch_size  = 4,
    gradient_checkpointing      = True, # recompute activations instead of storing them
                                        # ~30% slower but cuts activation memory ~60%

    # ── Generation length — biggest single memory lever ───────────────────
    generation_max_length       = 128,  # was 512; halving this saves a lot of VRAM


    # ── Optimizer memory reduction ────────────────────────────────────────
    optim                       = "adafactor",  # replaces Adam; uses ~10× less memory
                                                # no separate m/v buffers per parameter
    learning_rate               = FINETUNE_LEARNING_RATE,

    # ── Precision ─────────────────────────────────────────────────────────
    bf16                        = (DEVICE == 'cuda' and torch.cuda.is_bf16_supported()),
    fp16                        = (DEVICE == 'cuda' and not torch.cuda.is_bf16_supported()),

    # ── Rest unchanged ────────────────────────────────────────────────────
    predict_with_generate       = True,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    logging_steps               = 50,
    report_to                   = 'none',
)

# ── Also update inference generation length to match ──────────────────────
MAX_OUTPUT_LENGTH = 256  # keep consistent with generation_max_length above

# ── Trainer ───────────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model            = model_llm,
    args             = training_args,
    train_dataset    = hf_train_ds,
    eval_dataset     = hf_val_ds,
    processing_class = tokenizer,
    data_collator    = data_collator,
)

# Clear any cached memory before starting
torch.cuda.empty_cache()

print('Starting fine-tuning...')
print(f'  Training on {len(hf_train_ds):,} examples for {FINETUNE_EPOCHS} epoch(s)')
print(f'  Effective batch size: {1 * 8} (1 per device × 8 grad accumulation steps)')
trainer.train()

print('\n✅ Fine-tuning complete')


In [ ]:
# ── Regenerate test predictions with the fine-tuned model ─────────────────
# model_llm now holds the best fine-tuned checkpoint (load_best_model_at_end=True).
# We reuse generate_answers_batch() directly — it already uses model_llm
# and applies the same build_prompt() + sentinel-token cleanup.

print(f'Generating fine-tuned answers for {len(test):,} test questions...')
model_llm.eval()

test_questions_ft = test[TEST_QUESTION_COL].tolist()
test_languages_ft = test[TEST_LANG_COL].tolist() if TEST_LANG_COL else None

test_pred_finetuned = generate_answers_batch(test_questions_ft, test_languages_ft)

print(f'\n✅ Generated {len(test_pred_finetuned):,} answers')

# Preview
preview_ft = test[[TEST_ID_COL, TEST_QUESTION_COL]].copy()
preview_ft['finetuned_answer'] = test_pred_finetuned
display(preview_ft.head(5))

# ── Validate on val set before saving ─────────────────────────────────────
if compute_rouge:
    val_q_ft  = val[QUESTION_COL].tolist()
    val_l_ft  = val[LANG_COL].tolist() if LANG_COL else None
    val_ref_ft = val[ANSWER_COL].tolist()

    val_pred_ft = generate_answers_batch(val_q_ft, val_l_ft)
    metrics_ft  = compute_rouge(val_pred_ft, val_ref_ft)

    print(f'\n📊 Fine-tuned LLM — Validation ROUGE Scores')
    print(f'   ROUGE-1 F1 : {metrics_ft["rouge1_f1"]:.4f}')
    print(f'   ROUGE-L F1 : {metrics_ft["rougeL_f1"]:.4f}')

    if LANG_COL and LANG_COL in val.columns:
        print('\n📊 ROUGE scores by language (fine-tuned model):')
        lang_metrics_ft = compute_rouge_by_language(
            val_pred_ft, val_ref_ft, val[LANG_COL].tolist()
        )
        display(lang_metrics_ft.round(4))

# ── Save fine-tuned submission ─────────────────────────────────────────────
print('\nSaving fine-tuned submission...')
sub_finetuned = make_submission(
    ids         = test[TEST_ID_COL].values,
    predictions = test_pred_finetuned,
    output_path = OUTPUT_FINETUNED,
)
